# NumPy Reshape & Broadcasting

> 📘 **Python Mastery** · Module 10 — NumPy · Lesson 4/7

Real data never arrives in the shape you need. Reshaping rearranges the SAME values into new dimensions without copying, and broadcasting lets mismatched shapes work together — together they are the grammar of every ML preprocessing pipeline.

## 🎯 Learning Objectives

- Change array shapes with `reshape`, including the `-1` auto-dimension
- Choose between `ravel` (view) and `flatten` (copy)
- Transpose with `.T` and add axes using `np.newaxis` / `expand_dims`
- Combine and stack arrays with `concatenate`, `vstack`, `hstack` and `stack`
- Apply the broadcasting rules to predict which shapes are compatible
- Normalize dataset columns in one line using broadcasting

## 1. reshape: Same Data, New Shape

`reshape` reorganizes the existing elements into new dimensions — the VALUES stay in order, only their arrangement changes. The new shape must hold the same total number of elements.

**Syntax:**
```python
m = arr.reshape(3, 4)      # reinterpret as 3 rows x 4 columns
m = arr.reshape(4, -1)     # -1 means "you compute this dimension"
```

In [ ]:
import numpy as np

flat = np.arange(12)              # [0 1 2 ... 11]
grid = flat.reshape(3, 4)

print("original:", flat.shape)
print("reshaped:")
print(grid)
print("shares memory?", np.shares_memory(flat, grid), "- reshape usually returns a VIEW!")

In [ ]:
import numpy as np

readings = np.arange(24)

print("reshape(4, -1) ->", readings.reshape(4, -1).shape)   # -1 auto-fills: (4, 6)
print("reshape(-1, 8) ->", readings.reshape(-1, 8).shape)   # (3, 8)

images = readings.reshape(2, 3, 4)     # e.g. 2 grayscale images of 3x4 pixels
print("3-D batch      ->", images.shape)

back = images.reshape(-1)              # collapse everything back to 1-D
print("flattened      ->", back.shape)

## 2. ravel vs flatten: Two Ways to Go Flat

Both produce a 1-D version of an array. `ravel()` returns a **view** when it can (free), while `flatten()` always builds a fresh **copy** (safe). Pick based on whether you plan to edit the result.

**Syntax:**
```python
arr.ravel()      # 1-D view when possible
arr.flatten()    # 1-D copy, always independent
```

In [ ]:
import numpy as np

grid = np.arange(6).reshape(2, 3)

flat_view = grid.ravel()       # a window onto grid's memory
flat_view[0] = 99
print("after editing ravel() result:")
print(grid)                    # grid changed!

grid = np.arange(6).reshape(2, 3)   # start over
flat_copy = grid.flatten()
flat_copy[0] = 99
print("after editing flatten() result:")
print(grid)                    # grid untouched

## 3. Transpose: .T

`.T` flips rows and columns — shape `(r, c)` becomes `(c, r)`. Like reshape, it returns a view: no data is moved, NumPy just reads the memory in the other direction.

**Syntax:**
```python
m.T                      # attribute form
np.transpose(m)          # function form - identical
np.transpose(m, (1, 0))  # explicit axis permutation for higher dims
```

In [ ]:
import numpy as np

grades = np.array([[78, 85, 90],     # 2 students x 3 subjects
                   [62, 71, 88]])

print("original shape:", grades.shape)
print("transposed (subjects become rows):")
print(grades.T)
print("function form :", np.transpose(grades).shape)
print("still a view? ", np.shares_memory(grades, grades.T))

## 4. Adding Axes: np.newaxis, expand_dims, squeeze

Sometimes you need to *change rank*, not just sizes: turning a length-3 vector into a column of shape `(3, 1)` or a row of shape `(1, 3)`. This matters constantly — matrix multiplication and broadcasting both care about where the extra axis sits.

**Syntax:**
```python
v[:, np.newaxis]              # (3,) -> (3, 1) column vector
v[np.newaxis, :]              # (3,) -> (1, 3) row vector
np.expand_dims(v, axis=1)     # same as above, function form
np.squeeze(m)                 # remove all length-1 axes
```

In [ ]:
import numpy as np

v = np.array([1.0, 2.0, 3.0])
print("vector      :", v.shape)          # (3,)

col = v[:, np.newaxis]
row = v[np.newaxis, :]
print("as column   :", col.shape)
print(col)
print("as row      :", row.shape)

print("expand_dims :", np.expand_dims(v, axis=1).shape)   # (3, 1)
print("squeeze back:", np.squeeze(row).shape)             # (3,)

In [ ]:
import numpy as np

W = np.arange(6, dtype=float).reshape(2, 3)   # weights: (out_features, in_features)
x = np.array([1.0, 2.0, 3.0])                # one input sample: (in_features,)

print("W @ x          =", W @ x)                     # (2,3) @ (3,)  -> (2,)
X_batch = x[np.newaxis, :]                     # a BATCH of one sample: (1, 3)
print("batch matmul   =", X_batch @ W.T)             # (1,3) @ (3,2) -> (1, 2)
print("batch out shape:", (X_batch @ W.T).shape)

## 5. Stacking Arrays Together

Reshaping changes ONE array; stacking JOINS several. `concatenate` glues along an existing axis; `vstack`/`hstack` are friendly names for vertical/horizontal glue; `stack` creates a brand NEW axis (arrays become layers).

**Syntax:**
```python
np.concatenate([a, b], axis=0)   # join along an existing axis
np.vstack([a, b])                # stack as rows (axis 0)
np.hstack([a, b])                # join side by side (axis 1)
np.stack([a, b])                 # NEW axis: shape gains a layer dimension
```

In [ ]:
import numpy as np

week1 = np.array([31.5, 32.0, 33.8])
week2 = np.array([30.2, 29.9, 34.1])

print("vstack - each week becomes a row:")
print(np.vstack([week1, week2]))
print()
print("hstack - weeks glued end to end:", np.hstack([week1, week2]))
print("concatenate on 1-D equals hstack:", np.concatenate([week1, week2]).shape)

In [ ]:
import numpy as np

a = np.array([[1, 2],
              [3, 4]])
b = np.array([[5, 6],
              [7, 8]])

print("concatenate axis=0 (new rows):")
print(np.concatenate([a, b], axis=0))
print()
print("concatenate axis=1 (new columns):")
print(np.concatenate([a, b], axis=1))
print()
layers = np.stack([a, b])               # two matrices become one 3-D array
print("stacked shape:", layers.shape, "- layer 0:")
print(layers[0])

## 6. Broadcasting Rules

Broadcasting lets arrays of different shapes interact without copying data. NumPy compares the shapes **right-aligned**, dimension by dimension. Each pair of dimensions is compatible when they are **equal**, or when **one of them is 1** (it gets virtually stretched). Missing leading dimensions count as 1.

Worked examples against a `(3, 4)` matrix `M`:

| Other shape | Right-aligned check | Verdict |
|---|---|---|
| `(4,)` | 4 vs 4 on last axis | ✅ broadcasts → `(3, 4)` |
| `(3, 1)` | 3 vs 3, then 1 vs 4 | ✅ stretches → `(3, 4)` |
| `(1, 4)` | 1 vs 3, then 4 vs 4 | ✅ stretches → `(3, 4)` |
| `(3,)` | last axis: 4 vs 3 clash | ❌ ValueError |
| `scalar` | empty shape fits anywhere | ✅ everywhere |

**Syntax:**
```python
M + row_vector      # (3,4) + (4,)   works
M + column_vector   # (3,4) + (3,1)  works
M + scalar          # broadcast to every element
```

In [ ]:
import numpy as np

M = np.arange(12).reshape(3, 4)
print(M)
print()

bonus = np.array([10, 20, 30, 40])            # shape (4,) - one bonus per column
print("(3,4) + (4,)  ->", (M + bonus).shape)
print(M + bonus)
print()

col = np.array([[100], [200], [300]])         # shape (3, 1) - one per row
print("(3,4) + (3,1) ->", (M + col).shape)
print(M + col)

In [ ]:
import numpy as np

M = np.arange(12).reshape(3, 4)
vec = np.array([1, 2, 3])          # shape (3,)

try:
    M + vec                        # last axis: 4 vs 3 -> clash
except ValueError as err:
    print("ValueError:", err)

# Fix: give the vector a second axis so it aligns as (3, 1)
column_form = vec[:, np.newaxis]
print("as (3, 1) it now broadcasts:")
print(M + column_form)

> 🔍 **Under the Hood:** Broadcast stretching allocates NOTHING. NumPy fabricates the stretched array by setting that axis's stride to 0 — the loop re-reads the same memory address at every step. So `M + col` streams one 3-element vector three times instead of building a `(3, 4)` temporary. For the same reason `reshape` on a contiguous array merely rewrites header metadata (shape and strides): zero bytes move, which is why reshaping gigabyte arrays is effectively instant.

## 7. Real Use Case: Normalizing Columns with Broadcasting

ML preprocessing subtracts each feature's mean and divides by its standard deviation — a textbook one-liner once you combine `axis=0` aggregation with broadcasting. Each COLUMN is a feature, so stats come out with shape `(n_features,)` and broadcast down every row.

**Syntax:**
```python
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
```

In [ ]:
import numpy as np

# Four houses: (area_m2, bedrooms, price_lakh)
X = np.array([[1200, 3, 85],
              [1450, 4, 110],
              [900, 2, 60],
              [1600, 3, 130]], dtype=float)

means = X.mean(axis=0)      # one mean per column -> shape (3,)
stds = X.std(axis=0)
print("column means:", means)

X_norm = (X - means) / stds     # (4,3) minus (3,) broadcasts across every row
print("normalized (each column now has mean 0, std 1):")
print(np.round(X_norm, 2))

print("check means:", np.round(X_norm.mean(axis=0), 6),
      "| check stds:", np.round(X_norm.std(axis=0), 6))

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Reshaping to a shape with a different element count | `ValueError: cannot reshape array of size 12 into shape (5,2)` | Keep the product equal, or use `-1` for one free dimension |
| Assuming `reshape` copies data | It usually views — editing the reshaped array edits the original | `.copy()` if independence matters |
| Mixing up `flatten` and `ravel` | Mutating a `ravel` result can change the source | Use `flatten` when you will modify the flat array |
| Adding a `(3,)` vector to a `(3, 4)` matrix | Right-alignment fails: 4 vs 3 → `ValueError` | Insert an axis: `vec[:, np.newaxis]` → `(3, 1)` |
| Expecting `.T` to reverse a 1-D array | A vector's transpose is itself | Add an axis first: `v[:, np.newaxis]` |

✅ Debugging mantra: print `.shape` before every operation whose result surprises you.

## 💡 Best Practices & Pro Tips

- Allow exactly ONE `-1` per reshape — it is a wildcard, not a suggestion box.
- Prefer `concatenate([a, b], axis=...)` with an explicit axis; `vstack`/`hstack` get confusing once arrays exceed 2-D.
- Keep datasets 2-D as `(n_samples, n_features)` — scikit-learn, pandas and every tutorial assume it.
- Reach for `keepdims=True` on aggregations when the result must stay broadcast-ready.
- **AI-engineering relevance:** adding batch/channel axes (`(28, 28)` → `(1, 28, 28)`), computing attention as `Q @ K.T`, mean-normalizing inputs, and layer statistics all reduce to this lesson: reshape, transpose, broadcast.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `arr.reshape(r, c)` | Same data, new shape (usually a view) | `a.reshape(3, 4)` |
| `reshape(..., -1)` | Auto-compute one dimension | `a.reshape(4, -1)` |
| `arr.ravel()` | Flatten as a view when possible | free flattening |
| `arr.flatten()` | Flatten as an independent copy | safe flattening |
| `arr.T` | Transpose (rows ↔ columns) | `m.T` |
| `v[:, np.newaxis]` | Add an axis: vector → column | `(3,)` → `(3, 1)` |
| `np.expand_dims(a, axis)` | Function form of newaxis | `expand_dims(v, 1)` |
| `np.squeeze(a)` | Remove length-1 axes | `(1, 3)` → `(3,)` |
| `np.concatenate([a, b], axis)` | Join along an existing axis | `concatenate([a,b], 1)` |
| `np.vstack / hstack` | Row-wise / side-by-side glue | `np.vstack([w1, w2])` |
| `np.stack([a, b])` | Join along a NEW axis | two matrices → `(2, 2, 2)` |

Key takeaways:
- Reshape reinterprets memory; it does not duplicate it.
- Broadcasting compares shapes right-aligned: each dimension must match or be 1.
- `(3,4)+(4,)` ✅, `(3,4)+(3,1)` ✅, `(3,4)+(3,)` ❌ — remember which way the vector stands.
- Column normalization is the canonical broadcasting use-case: `(X - mean) / std`.

Broadcasting rule in one line: **right-align the shapes; every pair of dimensions must be equal or contain a 1.**

## 🔗 Next Lesson

- Continue to **[05_Array_Math_Aggregation](../05_Array_Math_Aggregation/notes.ipynb)** — ufuncs, aggregations, and finally demystifying `axis=0` versus `axis=1`.